# Flower Recommender System บน Neo4j (Google Colab)

โน้ตบุ๊กนี้แปลงระบบแนะนำดอกไม้ (จากไฟล์ `FlowerRecommenderSystem_10users_12flowers.ipynb` ที่ใช้ NetworkX) ให้ทำงานบน **Neo4j** จริง ตามแนวทางที่อาจารย์สอนไว้ในไฟล์ `PraceticeNeo4jDBOnline.ipynb`

**โครงสร้างกราฟ**
```
Node 2 ประเภท: User, Flower
Relationship: PREFERS (มีน้ำหนัก = คะแนนความชอบ 1-5)

(User)-[:PREFERS {rating}]->(Flower)
```

**รันบน Google Colab ได้ทันที** — แค่ต้องมี Neo4j AuraDB instance ของตัวเอง (ฟรี) แล้วใส่ URI / Username / Password ตอนรันเซลล์เชื่อมต่อ (เซลล์จะถามด้วย `getpass` เหมือนไฟล์ของอาจารย์ ไม่ต้องเขียนรหัสผ่านลงในโค้ด)

ถ้ายังไม่มี Aura instance: ไปที่ https://console.neo4j.io → สร้าง **AuraDB Free** instance ใหม่ → ระบบจะให้ดาวน์โหลดไฟล์ credentials (มี URI, Username, Password) เก็บไว้ให้ดี แล้วนำมาใส่ในเซลล์ด้านล่าง

## 1. ติดตั้งและเชื่อมต่อ Neo4j

In [ ]:
!pip install -q neo4j


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)


Neo4j Python Driver Version: 6.3.1


In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

# ใส่ URI และ Username ของ Aura instance ของตัวเอง (ดูได้จากไฟล์ credentials ที่ดาวน์โหลดตอนสร้าง instance)
#URI = input("Neo4j URI (เช่น neo4j+s://xxxxxxxx.databases.neo4j.io): ").strip()
#USERNAME = input("Username (เช่น neo4j): ").strip()
#PASSWORD = getpass("Password: ")
#dhzqnoySyTK5iK8bDxOO9_0X7A0XXhbXjCCDvFlGXrY

URI = "neo4j+s://dd4228f1.databases.neo4j.io"
USERNAME = "neo4j"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")


Password: ··········
Connected successfully


หา home database แล้วเก็บชื่อไว้ใช้ query ทุกคำสั่งถัดไป

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

DATABASE = result.records[0]["name"]

print("Database =", DATABASE)


Database = 69707966


In [ ]:
# ทดสอบใช้งานฐานข้อมูล
result = driver.execute_query(
    """
    RETURN 'Hello Neo4j' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])


Hello Neo4j


## 2. สร้าง Constraint

กำหนดให้ชื่อ User และชื่อ Flower ไม่ซ้ำกัน (เหมือนแนวทาง `student_id_unique` / `book_id_unique` ของอาจารย์)

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique
    IF NOT EXISTS

    FOR (u:User)

    REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

driver.execute_query(
    """
    CREATE CONSTRAINT flower_name_unique
    IF NOT EXISTS

    FOR (f:Flower)

    REQUIRE f.name IS UNIQUE
    """,
    database_=DATABASE
)

print("สร้าง constraint สำเร็จ")


สร้าง constraint สำเร็จ


## 3. โหลดข้อมูล User และ Flower เข้า Neo4j

ใช้ข้อมูลชุดเดียวกับไฟล์ `FlowerRecommenderSystem_10users_12flowers.ipynb` (10 ผู้ใช้, 12 สายพันธุ์ดอกไม้)

In [ ]:
users = [
    "Praew", "Kaew", "Miu", "Fon", "Palm",
    "Joy", "Top", "Nook", "Game", "Bow"
]


In [ ]:
flowers = [
    {"name": "Jasmine",       "meaning": "ความรักอันบริสุทธิ์ ความกตัญญู"},
    {"name": "Pink Rose",     "meaning": "ความรักอันอ่อนโยน ความขอบคุณ"},
    {"name": "Lotus",         "meaning": "ความบริสุทธิ์ การหลุดพ้นจากกิเลส"},
    {"name": "Sunflower",     "meaning": "ความหวัง พลังบวก"},
    {"name": "White Lily",    "meaning": "ความบริสุทธิ์ เกียรติยศ"},
    {"name": "Purple Orchid", "meaning": "ความสง่างาม เสน่ห์"},
    {"name": "Chrysanthemum", "meaning": "ความสัตย์ซื่อ อายุยืนยาว"},
    {"name": "Red Carnation", "meaning": "ความรักของแม่"},
    {"name": "Blue Iris",     "meaning": "ปัญญา ความหวัง ศรัทธา"},
    {"name": "Red Tulip",     "meaning": "ความรักที่สมบูรณ์แบบ"},
    {"name": "Daisy",         "meaning": "ความรักบริสุทธิ์ ความสดใสไร้เดียงสา"},
    {"name": "Hydrangea",     "meaning": "ความเข้าใจ การขอบคุณจากหัวใจ"}
]


In [ ]:
# ความสัมพันธ์: (user, flower, rating) — ชุดเดียวกับไฟล์ NetworkX เดิม
preferences = [
    {"user": "Praew", "flower": "Jasmine",       "rating": 5},
    {"user": "Praew", "flower": "Pink Rose",     "rating": 4},
    {"user": "Praew", "flower": "Lotus",         "rating": 3},

    {"user": "Kaew",  "flower": "Jasmine",       "rating": 4},
    {"user": "Kaew",  "flower": "Pink Rose",     "rating": 5},
    {"user": "Kaew",  "flower": "Sunflower",     "rating": 3},
    {"user": "Kaew",  "flower": "White Lily",    "rating": 4},

    {"user": "Miu",   "flower": "Lotus",         "rating": 5},
    {"user": "Miu",   "flower": "Purple Orchid", "rating": 4},
    {"user": "Miu",   "flower": "Chrysanthemum", "rating": 3},

    {"user": "Fon",   "flower": "Sunflower",     "rating": 5},
    {"user": "Fon",   "flower": "White Lily",    "rating": 4},
    {"user": "Fon",   "flower": "Blue Iris",     "rating": 4},
    {"user": "Fon",   "flower": "Red Tulip",     "rating": 3},

    {"user": "Palm",  "flower": "Purple Orchid", "rating": 5},
    {"user": "Palm",  "flower": "Chrysanthemum", "rating": 4},
    {"user": "Palm",  "flower": "Red Carnation", "rating": 5},

    {"user": "Joy",   "flower": "Daisy",         "rating": 5},
    {"user": "Joy",   "flower": "Pink Rose",     "rating": 4},
    {"user": "Joy",   "flower": "Jasmine",       "rating": 3},

    {"user": "Top",   "flower": "Daisy",         "rating": 4},
    {"user": "Top",   "flower": "Hydrangea",     "rating": 5},
    {"user": "Top",   "flower": "Sunflower",     "rating": 3},

    {"user": "Nook",  "flower": "Hydrangea",     "rating": 4},
    {"user": "Nook",  "flower": "Blue Iris",     "rating": 5},

    {"user": "Game",  "flower": "Red Tulip",     "rating": 4},
    {"user": "Game",  "flower": "Daisy",         "rating": 3},
    {"user": "Game",  "flower": "Red Carnation", "rating": 4},

    {"user": "Bow",   "flower": "Hydrangea",     "rating": 3},
    {"user": "Bow",   "flower": "White Lily",    "rating": 5},
    {"user": "Bow",   "flower": "Purple Orchid", "rating": 4}
]


สร้าง Node User, Node Flower และ Relationship `PREFERS` (ใช้ `MERGE` + `UNWIND` )

In [ ]:
result = driver.execute_query(
    """
    UNWIND $users AS name

    MERGE (u:User {name: name})

    RETURN count(u) AS total
    """,
    users=users,
    database_=DATABASE
)

print("จำนวน User ที่ประมวลผล =", result.records[0]["total"])


จำนวน User ที่ประมวลผล = 10


In [ ]:
result = driver.execute_query(
    """
    UNWIND $flowers AS row

    MERGE (f:Flower {name: row.name})

    SET f.meaning = row.meaning

    RETURN count(f) AS total
    """,
    flowers=flowers,
    database_=DATABASE
)

print("จำนวน Flower ที่ประมวลผล =", result.records[0]["total"])


จำนวน Flower ที่ประมวลผล = 12


In [ ]:
result = driver.execute_query(
    """
    UNWIND $preferences AS row

    MATCH
        (u:User {name: row.user}),
        (f:Flower {name: row.flower})

    MERGE (u)-[r:PREFERS]->(f)

    SET r.rating = row.rating

    RETURN count(r) AS total
    """,
    preferences=preferences,
    database_=DATABASE
)

print("จำนวน PREFERS ที่ประมวลผล =", result.records[0]["total"])


จำนวน PREFERS ที่ประมวลผล = 31


## 4. ตรวจสอบข้อมูลที่โหลดเข้าไป

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())


{'nodes': 22, 'relationships': 31}


In [ ]:
import pandas as pd

result = driver.execute_query(
    """
    MATCH (u:User)-[r:PREFERS]->(f:Flower)

    RETURN
        u.name AS user,
        f.name AS flower,
        r.rating AS rating

    ORDER BY user, rating DESC
    """,
    database_=DATABASE
)

pd.DataFrame([record.data() for record in result.records])


,user,flower,rating
0,Bow,White Lily,5
1,Bow,Purple Orchid,4
2,Bow,Hydrangea,3
3,Fon,Sunflower,5
4,Fon,White Lily,4
5,Fon,Blue Iris,4
6,Fon,Red Tulip,3
7,Game,Red Carnation,4
8,Game,Red Tulip,4
9,Game,Daisy,3


## 5. ตัวอย่าง Query พื้นฐาน

ดอกไม้ที่ Fon ชอบ พร้อมคะแนน (เทียบกับ `G.neighbors("Fon")` ในไฟล์ NetworkX เดิม)

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User {name: $name})-[r:PREFERS]->(f:Flower)

    RETURN f.name AS flower, r.rating AS rating

    ORDER BY rating DESC
    """,
    name="Fon",
    database_=DATABASE
)

for record in result.records:
    print(f'{record["flower"]} (ให้คะแนน {record["rating"]})')


Sunflower (ให้คะแนน 5)
White Lily (ให้คะแนน 4)
Blue Iris (ให้คะแนน 4)
Red Tulip (ให้คะแนน 3)


หา User ที่รสนิยมใกล้เคียงกับ Fon (ชอบดอกไม้เดียวกันอย่างน้อย 1 ชนิด)

In [ ]:
result = driver.execute_query(
    """
    MATCH (me:User {name: $name})-[:PREFERS]->(f:Flower)<-[:PREFERS]-(other:User)

    WHERE other <> me

    RETURN DISTINCT f.name AS flower, other.name AS similar_user

    ORDER BY flower
    """,
    name="Fon",
    database_=DATABASE
)

for record in result.records:
    print("ดอกไม้:", record["flower"], "-> ผู้ใช้ที่รสนิยมใกล้เคียง:", record["similar_user"])


ดอกไม้: Blue Iris -> ผู้ใช้ที่รสนิยมใกล้เคียง: Nook
ดอกไม้: Red Tulip -> ผู้ใช้ที่รสนิยมใกล้เคียง: Game
ดอกไม้: Sunflower -> ผู้ใช้ที่รสนิยมใกล้เคียง: Kaew
ดอกไม้: Sunflower -> ผู้ใช้ที่รสนิยมใกล้เคียง: Top
ดอกไม้: White Lily -> ผู้ใช้ที่รสนิยมใกล้เคียง: Kaew
ดอกไม้: White Lily -> ผู้ใช้ที่รสนิยมใกล้เคียง: Bow


## 6. Mini Project: Recommendation System

แนวคิด (ตรงกับฟังก์ชัน `recommend_flowers()` ในไฟล์ NetworkX เดิม แปลงเป็น Cypher):

```
target_user
 -> ดอกไม้ที่ target_user ชอบ
 -> User อื่นที่ชอบดอกไม้เดียวกัน
 -> ดอกไม้อื่นที่ User เหล่านั้นชอบ แต่ target_user ยังไม่เคยเลือก
 -> คะแนนแนะนำ = ผลรวมคะแนนของ user อื่นต่อดอกไม้นั้น
 -> เรียงคะแนนมาก -> น้อย แล้วแนะนำ พร้อมความหมายของดอกไม้
```

In [ ]:
def recommend_flowers(target_user):

    query = """
    MATCH (me:User {name: $target_user})-[:PREFERS]->(liked:Flower)
    MATCH (liked)<-[:PREFERS]-(similar:User)-[r:PREFERS]->(rec:Flower)

    WHERE similar <> me
      AND NOT (me)-[:PREFERS]->(rec)

    RETURN
        rec.name AS flower,
        rec.meaning AS meaning,
        sum(r.rating) AS score

    ORDER BY score DESC, flower
    """

    result = driver.execute_query(
        query,
        target_user=target_user,
        database_=DATABASE
    )

    return pd.DataFrame([record.data() for record in result.records])


In [ ]:
recommend_flowers("Praew")


,flower,meaning,score
0,Daisy,ความรักบริสุทธิ์ ความสดใสไร้เดียงสา,10
1,White Lily,ความบริสุทธิ์ เกียรติยศ,8
2,Sunflower,ความหวัง พลังบวก,6
3,Purple Orchid,ความสง่างาม เสน่ห์,4
4,Chrysanthemum,ความสัตย์ซื่อ อายุยืนยาว,3


แสดงคำแนะนำให้ครบทั้ง 10 คน (เหมือน loop `for user in users: show_recommendations(G, user)` ในไฟล์เดิม)

In [ ]:
def show_recommendations(target_user):

    print(f"=== แนะนำดอกไม้สำหรับ {target_user} ===")

    df = recommend_flowers(target_user)

    if df.empty:
        print("  ไม่มีดอกไม้ให้แนะนำเพิ่มเติม")
    else:
        for _, row in df.iterrows():
            print(f'  {row["flower"]}  (คะแนน {row["score"]:.0f})  -> ความหมาย: {row["meaning"]}')

    print()


for user in users:
    show_recommendations(user)


=== แนะนำดอกไม้สำหรับ Praew ===
  Daisy  (คะแนน 10)  -> ความหมาย: ความรักบริสุทธิ์ ความสดใสไร้เดียงสา
  White Lily  (คะแนน 8)  -> ความหมาย: ความบริสุทธิ์ เกียรติยศ
  Sunflower  (คะแนน 6)  -> ความหมาย: ความหวัง พลังบวก
  Purple Orchid  (คะแนน 4)  -> ความหมาย: ความสง่างาม เสน่ห์
  Chrysanthemum  (คะแนน 3)  -> ความหมาย: ความสัตย์ซื่อ อายุยืนยาว

=== แนะนำดอกไม้สำหรับ Kaew ===
  Daisy  (คะแนน 14)  -> ความหมาย: ความรักบริสุทธิ์ ความสดใสไร้เดียงสา
  Blue Iris  (คะแนน 8)  -> ความหมาย: ปัญญา ความหวัง ศรัทธา
  Hydrangea  (คะแนน 8)  -> ความหมาย: ความเข้าใจ การขอบคุณจากหัวใจ
  Lotus  (คะแนน 6)  -> ความหมาย: ความบริสุทธิ์ การหลุดพ้นจากกิเลส
  Red Tulip  (คะแนน 6)  -> ความหมาย: ความรักที่สมบูรณ์แบบ
  Purple Orchid  (คะแนน 4)  -> ความหมาย: ความสง่างาม เสน่ห์

=== แนะนำดอกไม้สำหรับ Miu ===
  Red Carnation  (คะแนน 10)  -> ความหมาย: ความรักของแม่
  Jasmine  (คะแนน 5)  -> ความหมาย: ความรักอันบริสุทธิ์ ความกตัญญู
  White Lily  (คะแนน 5)  -> ความหมาย: ความบริสุทธิ์ เกียรติยศ
  Pink Rose  (คะแนน 4)  -> ควา

## 7. Function เสริม: ค้นหา User และดอกไม้ที่เคยเลือก

In [ ]:
def find_user(name):

    result = driver.execute_query(
        """
        MATCH (u:User {name: $name})

        RETURN u.name AS name
        """,
        name=name,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบผู้ใช้"

    return result.records[0].data()


find_user("Top")


{'name': 'Top'}

In [ ]:
def preferred_flowers(name):

    result = driver.execute_query(
        """
        MATCH (u:User {name: $name})-[r:PREFERS]->(f:Flower)

        RETURN
            f.name AS flower,
            f.meaning AS meaning,
            r.rating AS rating

        ORDER BY rating DESC
        """,
        name=name,
        database_=DATABASE
    )

    return pd.DataFrame([record.data() for record in result.records])


preferred_flowers("Top")


,flower,meaning,rating
0,Hydrangea,ความเข้าใจ การขอบคุณจากหัวใจ,5
1,Daisy,ความรักบริสุทธิ์ ความสดใสไร้เดียงสา,4
2,Sunflower,ความหวัง พลังบวก,3


## 8. ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab เข้าไปแล้ว ให้เปิด Neo4j Aura Console -> Query แล้วลองรัน:

```
MATCH (n) RETURN n LIMIT 25
```

จะเห็นกราฟความสัมพันธ์ระหว่าง User และ Flower ที่โหลดเข้าไปทั้งหมด

## 9. ปิด Connection เมื่อใช้งานเสร็จ

In [ ]:
driver.close()

print("Neo4j connection closed")


Neo4j connection closed
